# ตรวจเกณฑ์ตัดสิน: วิธีเดียวกับ Baseline CNN แต่ margin มาจากความแปรปรวนของ ResNet เอง

In [1]:
import os, json
import numpy as np
import pandas as pd

# ---------- ตั้งค่า: โฟลเดอร์ผลและชื่อชุดของแต่ละโมเดล (แก้ให้ตรงกับที่รันจริง) ----------
CHECKS = {
    "ResNet18 3D": {
        "dir": "runs_resnet18_3D", "control": "stem_5",
        "cands": ["lr0.001_adamw0.001_do0.0_ls0.0", "lr0.001_adamw0.01_do0.5_ls0.05", "lr0.01_adam0.0001_do0.1_ls0.1",
                  "lr0.001_adamw0.0001_do0.0_ls0.15", "lr0.001_adam0.001_do0.5_ls0.05"],
    },
    "ResNet18 2D": {
        "dir": "runs_grid_2D", "control": "control",
        "cands": ["lr0.01_adam0.0001_do0.5_ls0.15", "lr0.001_adam0.01_do0.5_ls0.1", "lr0.001_adam0.0001_do0.5_ls0.15",
                  "lr0.001_adamw0.001_do0.1_ls0.05", "lr0.001_adam0.0001_do0.0_ls0.15"],
    },
    "ResNet18 2.5D": {
        "dir": "runs_grid_25D", "control": "control",
        "cands": ["lr0.001_adamw0.01_do0.3_ls0.15", "lr0.0001_adam0.001_do0.5_ls0.1", "lr0.0001_adam0.0001_do0.3_ls0.2",
                  "lr0.0001_adamw0.001_do0.3_ls0.0", "lr0.0001_adam0.01_do0.1_ls0.0", "lr0.0001_wd0_do0.0_ls0.0"],
    },
    "ResNet50 3D": {          # ยังไม่มี top-5: รอ grid เสร็จ แล้วใส่ชื่อชุดจาก top-5 ใน cands (ตอนนี้ขึ้นแค่ control)
        "dir": "runs_resnet50_3D", "control": "stem_7",
        "cands": [],
    },
    "ResNet50 2D": {
        "dir": "runs_grid50_2D", "control": "control",
        "cands": ["lr0.01_adamw0.0001_do0.3_ls0.05", "lr0.01_adamw0.0001_do0.1_ls0.2", "lr0.01_wd0_do0.0_ls0.0",
                  "lr0.01_adamw0.0001_do0.1_ls0.05", "lr0.01_adamw0.001_do0.0_ls0.1"],
    },
    "ResNet50 2.5D": {
        "dir": "runs_grid50_25D", "control": "control",
        "cands": ["lr0.01_adamw0.0001_do0.1_ls0.0", "lr0.001_adam0.001_do0.5_ls0.1", "lr0.001_adam0.001_do0.3_ls0.1",
                  "lr0.001_adamw0.001_do0.5_ls0.15", "lr0.001_adam0.001_do0.1_ls0.05"],
    },
}
MARGIN_MULT = 1.0      # margin = MARGIN_MULT x SD ข้าม seed ของ last10 AUC ของ control (ต่อโมเดล) ห้ามปรับตามผลที่อยากได้
LEARNED_LOSS = float(np.log(2))   # "เรียนได้" = train loss เฉลี่ย 10 epoch สุดท้ายต่ำกว่าเดาสุ่ม (ln 2 = 0.693) Baseline ใช้ 0.66


In [2]:
def load(d, name):
    out = {}
    p = os.path.join(d, name, "results")
    if os.path.isdir(p):
        for f in os.listdir(p):
            if f.startswith("seed") and f.endswith(".json"):
                r = json.load(open(os.path.join(p, f)))
                out[int(r["seed"])] = r
    return out

def last10_auc(r):  return float(np.mean(r["history"]["val_auc"][-10:]))
def train10(r):     return float(np.mean(r["history"]["train_loss"][-10:]))

def sd(v):
    v = np.asarray(v, float)
    return float(v.std(ddof=1)) if len(v) > 1 else float("nan")

rows_all = {}
for model, cfg in CHECKS.items():
    d, ctl_name = cfg["dir"], cfg["control"]
    if not os.path.isdir(d):
        print(f"[{model}] ไม่พบโฟลเดอร์ {d} ข้ามไป"); continue
    ctl = load(d, ctl_name)
    if not ctl:
        print(f"[{model}] ไม่พบผลของ control ({ctl_name}) ข้ามไป"); continue
    ctl_sd = sd([last10_auc(r) for r in ctl.values()])       # ความแกว่งของ control เอง (ทุก seed ของ control)
    margin = MARGIN_MULT * ctl_sd
    rows = []
    for name in [ctl_name] + cfg["cands"]:
        runs = load(d, name)
        seeds = sorted(set(runs) & set(ctl))                 # เทียบเฉพาะ seed ที่มีทั้งสองชุด ใช้ทุก seed ที่มี
        if not seeds:
            continue
        n = len(seeds)
        l10 = np.array([last10_auc(runs[s]) for s in seeds]); c10 = np.array([last10_auc(ctl[s]) for s in seeds])
        t10 = np.array([train10(runs[s]) for s in seeds])
        g = lambda dd, k, m: np.array([dd[s][k][m] for s in seeds])
        diff = l10.mean() - c10.mean()
        sd_max = np.nanmax([sd(l10), sd(c10)]) if n > 1 else float("nan")
        is_ctl = name == ctl_name
        learned = bool(t10.mean() < LEARNED_LOSS)
        rows.append({
            "ชุด": "CONTROL" if is_ctl else name, "n seed": n,
            "last10 AUC": l10.mean(), "SD": sd(l10), "diff vs control": diff,
            "Δ/SD": diff / sd_max if sd_max == sd_max and sd_max > 0 else float("nan"),
            "train10 loss": t10.mean(), "learned": "ใช่" if learned else "ไม่",
            "margin": margin,
            "candidate (diff > margin และ learned)": "-" if is_ctl else ("ผ่าน" if (margin == margin and diff > margin and learned) else "ไม่ผ่าน"),
            "AUC last": g(runs, "val_last", "AUC").mean(), "F1 last": g(runs, "val_last", "F1").mean(),
            "AUC best (ประกอบ)": g(runs, "val_best", "AUC").mean(), "F1 best (ประกอบ)": g(runs, "val_best", "F1").mean(),
        })
    df = pd.DataFrame(rows)
    rows_all[model] = df
    print("=" * 100); print(f"{model} | โฟลเดอร์ {d} | control = {ctl_name} ({len(ctl)} seed) | SD ของ control {ctl_sd:.3f} -> margin {margin:.3f} | LEARNED_LOSS {LEARNED_LOSS:.3f}")
    print("=" * 100)
    display(df.round(3))
    win = df[df["candidate (diff > margin และ learned)"] == "ผ่าน"]
    if len(win):
        print("ผ่านเกณฑ์ (เป็นผู้เข้ารอบ):", win["ชุด"].tolist())
        for _, r in win.iterrows():
            print(f"   {r['ชุด']}: last10 AUC {r['diff vs control']:+.3f} (margin {r['margin']:.3f}) = {r['Δ/SD']:.2f} เท่าของ SD ข้าม seed -> "
                  + ("ส่วนต่างใหญ่กว่า SD" if r["Δ/SD"] >= 1 else "ส่วนต่างเล็กกว่า SD ต้องรายงานว่าอยู่ในช่วงความแปรปรวนข้าม seed"))
    else:
        print("ไม่มีชุดไหนผ่านเกณฑ์ -> คง control")
    print()


ResNet18 3D | โฟลเดอร์ runs_resnet18_3D | control = stem_5 (10 seed) | SD ของ control 0.049 -> margin 0.049 | LEARNED_LOSS 0.693


,ชุด,n seed,last10 AUC,SD,diff vs control,Δ/SD,train10 loss,learned,margin,candidate (diff > margin และ learned),AUC last,F1 last,AUC best (ประกอบ),F1 best (ประกอบ)
0,CONTROL,10,0.726,0.049,0.000,0.000,0.017,ใช่,0.049,-,0.736,0.640,0.803,0.764
1,lr0.001_adamw0.001_do0.0_ls0.0,10,0.745,0.031,0.019,0.380,0.023,ใช่,0.049,ไม่ผ่าน,0.745,0.648,0.829,0.762
2,lr0.001_adamw0.01_do0.5_ls0.05,5,0.722,0.054,-0.007,-0.137,0.122,ใช่,0.049,ไม่ผ่าน,0.723,0.668,0.813,0.763
3,lr0.01_adam0.0001_do0.1_ls0.1,5,0.686,0.062,-0.043,-0.697,0.263,ใช่,0.049,ไม่ผ่าน,0.685,0.471,0.815,0.702
4,lr0.001_adamw0.0001_do0.0_ls0.15,5,0.723,0.049,-0.007,-0.133,0.272,ใช่,0.049,ไม่ผ่าน,0.738,0.623,0.811,0.743
5,lr0.001_adam0.001_do0.5_ls0.05,5,0.691,0.039,-0.038,-0.767,0.191,ใช่,0.049,ไม่ผ่าน,0.670,0.618,0.814,0.737


ไม่มีชุดไหนผ่านเกณฑ์ -> คง control

[ResNet18 2D] ไม่พบโฟลเดอร์ runs_grid_2D ข้ามไป
[ResNet18 2.5D] ไม่พบโฟลเดอร์ runs_grid_25D ข้ามไป
[ResNet50 3D] ไม่พบโฟลเดอร์ runs_resnet50_3D ข้ามไป
[ResNet50 2D] ไม่พบโฟลเดอร์ runs_grid50_2D ข้ามไป
[ResNet50 2.5D] ไม่พบโฟลเดอร์ runs_grid50_25D ข้ามไป
